# PROYECTO INTEGRADOR · VISIÓN COMPUTACIONAL CON OPENCV
**Programa:** Python y visión computacional con OpenCV · **Instructor del material original:** Ing. Gunar Castro Iñigo

**Actividad final de la guía:** `GUIA_ENTREGA_ESTUDIANTE.md`, del archivo `SENATI_Vision_Computacional.rar`.

**Problema:** un equipo necesita transformar una fotografía para resaltar bordes y adaptar su tamaño para un reporte.

**Objetivo:** leer → comprobar → escala de grises → suavizado → bordes Canny → transformaciones → comparar → exportar.

**Entregables:** (a) seis PNG en `resultados/caso_integrador/`, (b) captura comparativa de las seis imágenes, (c) informe de una página y (d) comparación del reto 30° / Canny 100–200.

**Aplicación web:** [Vision Lab SENATI](https://vision-lab-senati-actividades.onrender.com/) · **Repositorio:** https://github.com/Loren07-kie/Actividades-ml-y-dp

Este cuaderno tiene un modo ilustrativo cuando no se dispone de `recursos/imagen_practica.png`. Para la **entrega fiel a SENATI**, usa la imagen original incluida en el RAR, de 900 × 560 píxeles; la ilustración generada automáticamente no debe presentarse como evidencia de la fotografía original.

## Correspondencia con la guía oficial de entrega

| Requisito de la guía | Solución en este cuaderno |
|---|---|
| Ejecutar `06_Caso_Integrador/07_Caso_Practico_Completo.ipynb` | Flujo reproducido y explicado aquí; también es recomendable ejecutar el cuaderno original |
| Imagen original, grises, filtro, bordes, reducida y rotada | Se generan como seis PNG en `resultados/caso_integrador/` |
| Informe de una página y nombres | Texto redactado al final; completar integrantes |
| Captura de seis imágenes | `comparacion_6_resultados.png` generada con Matplotlib |
| Rotación 30° y Canny 100/200 | Celdas con comparación de parámetros |
| Preguntas de cierre | Se responden en la sección final |

**Importante:** no confundas el examen adicional de la web con los entregables oficiales: el cuestionario es un refuerzo, no una exigencia de la guía.

## 1. Preparación del entorno
Si utilizas Jupyter local, instala con `pip install opencv-python numpy matplotlib pillow`. En Google Colab también funcionan estas bibliotecas.

In [ ]:
from pathlib import Path
import cv2, numpy as np, matplotlib.pyplot as plt
from PIL import Image
OUT=Path("resultados/caso_integrador"); OUT.mkdir(parents=True,exist_ok=True)
print("OpenCV:",cv2.__version__)

## 2. Lectura y verificación de la imagen
Utilizamos la imagen del material de clase, de 900 × 560 píxeles. Si no está disponible, generamos una escena de ejemplo para que el cuaderno siga siendo ejecutable.

In [ ]:
ruta=Path("recursos/imagen_practica.png")
USANDO_IMAGEN_ORIGINAL=ruta.exists()
if USANDO_IMAGEN_ORIGINAL:
    imagen=cv2.imread(str(ruta))
else:
    imagen=np.full((560,900,3),(236,225,208),np.uint8)
    cv2.rectangle(imagen,(100,120),(365,390),(65,130,245),-1)
    cv2.circle(imagen,(620,255),135,(45,190,90),-1)
    cv2.putText(imagen,"DEMOSTRACION",(110,505),cv2.FONT_HERSHEY_SIMPLEX,1.3,(45,45,45),3)
    print("ATENCIÓN: no se encontró recursos/imagen_practica.png. Los resultados son DEMOSTRATIVOS, no la evidencia oficial.")
assert imagen is not None
h,w=imagen.shape[:2]
print(f"Imagen: ancho {w} px, alto {h} px | Fuente oficial: {USANDO_IMAGEN_ORIGINAL}")
cv2.imwrite(str(OUT/"original.png"),imagen)
plt.imshow(cv2.cvtColor(imagen,cv2.COLOR_BGR2RGB));plt.axis("off");

## 3. Procesamiento y transformación
Escala de grises: reduce de tres canales a uno. Gauss 5×5: disminuye ruido. Canny 50/150: detecta contornos. Redimensionar 50% y rotar 15°.

In [ ]:
gris=cv2.cvtColor(imagen,cv2.COLOR_BGR2GRAY)
filtro=cv2.GaussianBlur(gris,(5,5),0)
bordes=cv2.Canny(filtro,50,150)
reducida=cv2.resize(imagen,(w//2,h//2),interpolation=cv2.INTER_AREA)
M=cv2.getRotationMatrix2D((w/2,h/2),15,1)
rotada=cv2.warpAffine(imagen,M,(w,h))
resultados={"grises":gris,"filtro":filtro,"bordes":bordes,"reducida":reducida,"rotada":rotada}
for nombre,img in resultados.items():
    cv2.imwrite(str(OUT/f"{nombre}.png"),img)
print("Seis entregables PNG generados:",*[p.name for p in OUT.glob("*.png") if p.name in ["original.png","grises.png","filtro.png","bordes.png","reducida.png","rotada.png"]],sep="\n• ")

## 4. Comparación visual de las seis imágenes

In [ ]:
fig,ax=plt.subplots(2,3,figsize=(15,8))
for a,(nombre,im) in zip(ax.flat,[("Original",imagen),("Grises",gris),("Suavizado",filtro),("Canny 50/150",bordes),("Reducida",reducida),("Rotada 15°",rotada)]):
    if im.ndim==2:a.imshow(im,cmap="gray",vmin=0,vmax=255)
    else:a.imshow(cv2.cvtColor(im,cv2.COLOR_BGR2RGB))
    a.set_title(nombre);a.axis("off")
plt.suptitle("Comparación del caso práctico integrador",fontsize=15)
plt.tight_layout();plt.savefig(OUT/"comparacion_6_resultados.png",dpi=160,bbox_inches="tight");plt.show()
print("Captura lista:",OUT/"comparacion_6_resultados.png")

## 5. Reto adicional: rotación 30° y Canny 100–200
Con 30° la orientación cambia más y normalmente aumentan zonas de relleno. Umbrales 100/200 omiten bordes de bajo contraste.

In [ ]:
M30=cv2.getRotationMatrix2D((w/2,h/2),30,1)
rot30=cv2.warpAffine(imagen,M30,(w,h))
canny_extra=cv2.Canny(filtro,100,200)
cv2.imwrite(str(OUT/"rotada_30.png"),rot30)
cv2.imwrite(str(OUT/"canny_100_200.png"),canny_extra)
print("Bordes detectados Canny 50/150:",np.count_nonzero(bordes))
print("Bordes detectados Canny 100/200:",np.count_nonzero(canny_extra))

## 6. Retos resueltos de los módulos 1 a 6 y ampliación
1. **Python:** al elevar el mínimo a 150 cambian 15, 80, 110; 200 permanece intacto.
2. **Lectura/visualización:** el recorte alternativo de 250 filas por 350 columnas muestra otra zona de la imagen.
3. **Matrices:** el rectángulo entre filas 80–139 y columnas 600–849 mide 60 × 250 píxeles.
4. **RGB/grises:** umbral 70 muestra más píxeles blancos que umbral 200.
5. **Filtros:** Gauss 9×9 reduce más detalles; Canny 100–200 conserva los bordes de mayor contraste.
6. **Geometría:** rotación −25° gira en sentido horario y desplazamiento x=−60 mueve la escena a la izquierda.
7. **Ampliación:** en BGR, rojo (0,0,255) cambia a verde (0,255,0) sin alterar la forma.

In [ ]:
# Comprobación reproducible de retos de umbrales, filtros y transformaciones
_,u70=cv2.threshold(gris,70,255,cv2.THRESH_BINARY)
_,u200=cv2.threshold(gris,200,255,cv2.THRESH_BINARY)
g9=cv2.GaussianBlur(gris,(9,9),0)
Mneg=cv2.getRotationMatrix2D((w/2,h/2),-25,1)
rot_neg=cv2.warpAffine(imagen,Mneg,(w,h))
T=np.float32([[1,0,-60],[0,1,0]])
mov=cv2.warpAffine(imagen,T,(w,h))
for nombre,im in {"umbral_70":u70,"umbral_200":u200,"gauss_9x9":g9,"rotacion_menos25":rot_neg,"desplazamiento_menos60":mov}.items():cv2.imwrite(str(OUT/f"{nombre}.png"),im)
print("Píxeles blancos: umbral 70",np.count_nonzero(u70),"| umbral 200",np.count_nonzero(u200))

## 7. Preguntas de cierre respondidas
**1. ¿Diferencia entre filtro y bordes?** Un filtro modifica intensidades o reduce ruido; Canny encuentra transiciones fuertes y representa contornos.

**2. ¿Por qué escala de grises?** Reduce la imagen a un canal de intensidad y simplifica el análisis de bordes.

**3. ¿Por qué aparecen zonas vacías tras rotar?** Algunos píxeles destino quedan sin correspondencia; OpenCV por defecto los rellena de negro.

**4. ¿Cuál técnica fue más útil?** Canny, para identificar límites de objetos. El filtro Gaussiano previo ayuda a reducir contornos espurios.

**Conclusión:** las operaciones permiten distinguir estructura visual, reducir ruido y comparar transformaciones. Conviene registrar parámetros y visualizar antes/después.

**Evidencias:** revisar la carpeta `resultados/` generada al ejecutar el notebook.

## Informe breve de una página — texto para entrega

**Título:** Aplicación de OpenCV al procesamiento de una imagen digital.

**Integrantes:** [Completar los nombres antes de entregar].

**Objetivo.** Procesar una imagen digital con OpenCV para mejorar su visualización, identificar contornos y aplicar transformaciones geométricas.

**Procedimiento y resultados.** (1) La imagen se abrió mediante `cv2.imread()` desde la carpeta `recursos` y se comprobó que la lectura fuese válida. (2) La imagen de práctica original tiene **900 píxeles de ancho y 560 píxeles de alto** (si se utiliza el archivo del RAR; comprobar con `shape`). (3) La conversión con `cv2.cvtColor(..., COLOR_BGR2GRAY)` redujo la información de color a intensidades de un canal, suficiente para detectar variaciones de brillo. (4) El suavizado Gaussiano de 5×5 atenuó ruido y pequeñas variaciones para facilitar la detección de contornos. (5) Canny, inicialmente con umbrales 50 y 150, identificó cambios fuertes de intensidad y produjo una imagen de bordes. (6) Las transformaciones geométricas consistieron en reducir ancho y alto a la mitad y girar la imagen 15° alrededor de su centro. Se exportaron los seis PNG requeridos.

**Reto adicional.** Al girar la imagen 30°, cambió su orientación y pueden aparecer más zonas vacías en las esquinas. Al utilizar Canny con umbrales 100 y 200 se priorizaron bordes de mayor contraste; los resultados se contrastaron con los parámetros iniciales.

**Interpretación y conclusión.** Para resaltar formas, la detección Canny fue especialmente útil, mientras que el suavizado ayudó a controlar los detalles no deseados. El procesamiento demostró que distintas operaciones modifican propiedades específicas de una imagen. Como evidencia, se adjunta la captura `comparacion_6_resultados.png`.

*Nota: si ejecutas este cuaderno con la imagen demostrativa, no presentes ese resultado como si perteneciera al archivo original del docente.*

In [ ]:
# Verificación automática de la lista de entrega
obligatorios=["original.png","grises.png","filtro.png","bordes.png","reducida.png","rotada.png","comparacion_6_resultados.png","rotada_30.png","canny_100_200.png"]
for f in obligatorios:
    p=OUT/f
    print(("✓" if p.exists() else "✗"),str(p))
assert all((OUT/f).exists() for f in obligatorios),"Faltan archivos: ejecuta las celdas en orden"
print("Nota: la evidencia oficial requiere la imagen_practica.png real:",USANDO_IMAGEN_ORIGINAL)